#### >> Utilizando a extensão jupyter (.ipynb)

In [1]:
import os
import polars as pl 
import matplotlib.pyplot as plt 
from datetime import datetime

pl.Config.set_tbl_rows(-1)
pl.Config.set_decimal_separator(',')
pl.Config.set_thousands_separator('.')
pl.Config.set_float_precision(2)

ENDERECO_DADOS = './../../DADOS/BOLSA_FAMILIA/'
ENDERECO_VOTACAO = './../../DADOS/VOTACAO/'

##### >> Obtendo os dados do arquivo

In [2]:

try:
    print('Lendo o arquivo...')

    df_bolsa_familia = pl.scan_parquet(ENDERECO_DADOS + 'bolsa_familia.parquet')   
    # print(df_bolsa_familia.head(5).collect())   

    df_dados_votacao = pl.read_csv(ENDERECO_VOTACAO + 'votacao_secao_2022_BR.csv', separator=';', encoding='iso-8859-1')
    # print(df_dados_votacao)
    # print(df_votacao.describe())
    # for coluna in df_dados_votacao.columns:
    #     print(coluna)
    
except Exception as e:
    print(f'Erro ao ler o arquivo: {e}')

Lendo o arquivo...


#### >> Filtrar p/segundo turno 'NR_TURNO' e 'NR_VOTAVEL' 13 e 22

In [3]:

try:
    print('Filtrando os dados para o segundo turno e candidatos 13 e 22...')
    df_turno02 = df_dados_votacao.filter(
    (pl.col('NR_TURNO') == 2) & 
    (pl.col('NR_VOTAVEL').is_in([13, 22]))
    )

    print('Dados filtrados com sucesso!')

except Exception as e:
    print(f'Erro ao filtrar os dados: {e}')

Filtrando os dados para o segundo turno e candidatos 13 e 22...
Dados filtrados com sucesso!


#### Processando Votação 

In [5]:
try:    
    print('Agrupando os dados...')

    # >> Delimitar  as variaveis 'SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'    
    df_votacao =df_turno02.lazy().select(['SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'])
    # print(df_votacao.head(5))

    # >> Converter para Categorical
    df_votacao = df_votacao.with_columns([
        pl.col('SG_UF').cast(pl.Categorical),
        pl.col('NM_VOTAVEL').cast(pl.Categorical)
    ])
    # print(df_votacao.head(5))

    # Agrupar/totalizar
    df_votacao = (
    df_votacao
    .group_by(["SG_UF", "NM_VOTAVEL"])
    .agg(
        pl.col("QT_VOTOS").sum()
    )
    .sort("SG_UF")
    )

    # coletar os dados
    df_votacao = df_votacao.collect()
    display(df_votacao) # >> print do Jupyter
    # print(df_votacao.head(5)) # >> print do terminal 

except Exception as e:
    print(f'Erro ao agrupar os dados: {e}')

Agrupando os dados...


In [7]:
# Processamento Bolsa Família
try:
    # Delimitar as variáveis, converter Categorical, Agrupar/totalizar

    # delimitando as vaiáveis 'UF', 'VALOR PARCELA'
    df_bolsa_familia = df_bolsa_familia.lazy().select(['UF', 'VALOR PARCELA'])

    # Converter Categorical
    df_bolsa_familia = df_bolsa_familia.with_columns(
        pl.col('UF').cast(pl.Categorical)
    )

    # Agrupar/totalizar
    df_bolsa_familia = (
        df_bolsa_familia.group_by('UF')
        .agg(pl.col('VALOR PARCELA').sum())
        .sort('UF', descending=False)
    )

    # coletar os dados
    df_bolsa_familia = df_bolsa_familia.collect()
    display(df_bolsa_familia)  # print do jupyter

except Exception as e:
    print(f'Erro ao processar dados de bolsa família: {e}')

UF,VALOR PARCELA
cat,f64
"""AC""","445.278.628,00"
"""AL""","1.719.485.364,00"
"""AM""","2.173.681.249,00"
"""AP""","404.493.438,00"
"""BA""","7.775.720.895,00"
"""CE""","4.496.280.805,00"
"""DF""","532.890.182,00"
"""ES""","939.327.731,00"
"""GO""","1.486.309.200,00"
